In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder

In [3]:
# load the data
df = sns.load_dataset('titanic')
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [4]:
# Numerical columns
df['age'] = df['age'].fillna(df['age'].median())
df['fare'] = df['fare'].fillna(df['fare'].median())

# Categorical columns
if not df['embark_town'].mode().empty:
    df['embark_town'] = df['embark_town'].fillna(df['embark_town'].mode().iloc[0])

if not df['embarked'].mode().empty:
    df['embarked'] = df['embarked'].fillna(df['embarked'].mode().iloc[0])

# Drop deck column
df.drop('deck', axis=1, inplace=True)

In [5]:
# Encode allcategorical variables and object variable
le_sex = LabelEncoder()
le_embarked = LabelEncoder()
le_embark_town = LabelEncoder()
le_class = LabelEncoder()
le_who = LabelEncoder()
le_adult_male = LabelEncoder()
le_alive = LabelEncoder()

df['sex'] = le_sex.fit_transform(df['sex'])
df['embarked'] = le_embarked.fit_transform(df['embarked'])
df['embark_town'] = le_embark_town.fit_transform(df['embark_town'])
df['class'] = le_class.fit_transform(df['class'])
df['who'] = le_who.fit_transform(df['who'])
df['adult_male'] = le_adult_male.fit_transform(df['adult_male'])
df['alive'] = le_alive.fit_transform(df['alive'])
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,1,22.0,1,0,7.2500,2,2,1,1,2,0,False
1,1,1,0,38.0,1,0,71.2833,0,0,2,0,0,1,False
2,1,3,0,26.0,0,0,7.9250,2,2,2,0,2,1,True
3,1,1,0,35.0,1,0,53.1000,2,0,2,0,2,1,False
4,0,3,1,35.0,0,0,8.0500,2,2,1,1,2,0,True


In [9]:
# Inverse transform the data
df['sex'] = le_sex.fit_transform(df['sex'])
df['embarked'] = le_embarked.inverse_transform(df['embarked'])
df['embark_town'] = le_embark_town.inverse_transform(df['embark_town'])
df['class'] = le_class.inverse_transform(df['class'])
df['who'] = le_who.inverse_transform(df['who'])
df['adult_male'] = le_adult_male.inverse_transform(df['adult_male'])
df['alive'] = le_alive.inverse_transform(df['alive'])
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,1,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False
1,1,1,0,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False
2,1,3,0,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True
3,1,1,0,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False
4,0,3,1,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True


# One-Hot Encode

In [11]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# Select categorical columns
categorical_cols = ['sex', 'embarked', 'embark_town', 
                    'class', 'who', 'adult_male', 'alive']

# Create encoder
ohe = OneHotEncoder(sparse_output=False)

# Fit and transform
encoded_data = ohe.fit_transform(df[categorical_cols])

# Create dataframe of encoded columns
encoded_df = pd.DataFrame(
    encoded_data,
    columns=ohe.get_feature_names_out(categorical_cols)
)

# Reset index to avoid mismatch
encoded_df.index = df.index

# Drop original categorical columns
df = df.drop(categorical_cols, axis=1)

# Concatenate encoded columns
df = pd.concat([df, encoded_df], axis=1)

# Display dataframe
df.head()

,survived,pclass,age,sibsp,parch,fare,alone,sex_0,sex_1,embarked_C,...,class_First,class_Second,class_Third,who_child,who_man,who_woman,adult_male_False,adult_male_True,alive_no,alive_yes
0,0,3,22.0,1,0,7.2500,False,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
1,1,1,38.0,1,0,71.2833,False,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
2,1,3,26.0,0,0,7.9250,True,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
3,1,1,35.0,1,0,53.1000,False,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
4,0,3,35.0,0,0,8.0500,True,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0


# Ordianl Encoder

In [13]:
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder

# Reload dataset
df = sns.load_dataset('titanic')

# Handle missing values
df['age'] = df['age'].fillna(df['age'].median())
df['fare'] = df['fare'].fillna(df['fare'].median())

df['embark_town'] = df['embark_town'].fillna(df['embark_town'].mode().iloc[0])
df['embarked'] = df['embarked'].fillna(df['embarked'].mode().iloc[0])

# Drop deck column
df.drop('deck', axis=1, inplace=True)

# Categorical columns
categorical_cols = [
    'sex',
    'embarked',
    'embark_town',
    'class',
    'who',
    'adult_male',
    'alive'
]

# Create encoder
oe = OrdinalEncoder()

# Encode
df[categorical_cols] = oe.fit_transform(df[categorical_cols])

# Show dataframe
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,1.0,22.0,1,0,7.2500,2.0,2.0,1.0,1.0,2.0,0.0,False
1,1,1,0.0,38.0,1,0,71.2833,0.0,0.0,2.0,0.0,0.0,1.0,False
2,1,3,0.0,26.0,0,0,7.9250,2.0,2.0,2.0,0.0,2.0,1.0,True
3,1,1,0.0,35.0,1,0,53.1000,2.0,0.0,2.0,0.0,2.0,1.0,False
4,0,3,1.0,35.0,0,0,8.0500,2.0,2.0,1.0,1.0,2.0,0.0,True
